# Pelagic spin-up validation notebook

This notebook follows the same structured workflow as `benthic_validation_spinup.ipynb`:

1. **Configuration** for paths, variables, layers, and output folders.
2. **Helper functions** for loading model output and handling grid/time quirks.
3. **Analysis functions** for trends, subdomain diagnostics, observations, and satellite comparison.
4. **Entry-point examples** that can be toggled on when you want to run a specific workflow.

The main cleanup change is that map-based diagnostics now **export individual PNG frames** instead of rendering animations inline in the notebook.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from IPython.display import display


In [ ]:
# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

POSTPROC_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
BASE_OUTPUT_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/')
TOPO_FILE = Path('/export/lv9/projects/dws/model_input/bathymetry/topo_dws_500m.nc')
SATELLITE_FILE = POSTPROC_DIR / 'satellite' / 'copernicus_satellite_chl_2015.nc'

DEFAULT_SPINUP = 'spinup_05'
SPINUP_NAMES = ['spinup_01', 'spinup_02', 'spinup_03', 'spinup_04', 'spinup_05']
YEAR = 2015
NC_PATTERN = 'dws_500m.3d.{year}??.nc'

MODEL_SURFACE_LAYER_INDEX = 10  # existing notebook convention: top=11, bottom=1
CHLA_DIAGNOSTIC_LAYERS = {
    'center': 5,
    'upper': 10,
    'lower': 1,
}

SUBDOMAIN = {
    'name': 'Marsdiep',
    'x_slice': (75, 78),
    'y_slice': (95, 98),
}

MEASUREMENT_FILE = POSTPROC_DIR / 'Jetty_HWseries.csv'
PRIMARY_PRODUCTION_FILE = POSTPROC_DIR / 'PP_HW40_2012_2023.csv'

TREND_VARIABLES = [
    'elev',
    'temp',
    'salt',
    'O2o',
    'netPPm2',
    'N1p',
    'N3n',
    'N4n',
    'N5s',
    'N6r',
    'P1c',
    'P2c',
    'P3c',
    'P4c',
    'P5c',
    'P6c',
    'Chla',
    'Y1c',
    'Y2c',
    'Y3c',
    'Y5c',
    'Yy3c',
]

NONNEGATIVE_VARS = {'Chla', 'netPPm2'}
OBS_VAR_MAP = {
    'Chla': 'Chl',
    'N1p': 'PO4',
    'N3n': 'NO3',
    'N4n': 'NH4',
}

FRAME_OUTPUT_DIR = POSTPROC_DIR / DEFAULT_SPINUP / f'{YEAR}_spinup_diagnostics' / 'frames'
FRAME_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FRAME_EXPORTS = [
    {
        'plotvar': 'elev',
        'output_dir': FRAME_OUTPUT_DIR / 'elev',
        'layer_index': None,
        'time_step': 1,
        'cmap': 'viridis',
        'vmin': -1.5,
        'vmax': 1.5,
        'mask_var': None,
        'mask_threshold': None,
    },
    {
        'plotvar': 'Chla',
        'output_dir': FRAME_OUTPUT_DIR / 'chla_surface',
        'layer_index': MODEL_SURFACE_LAYER_INDEX,
        'time_step': 1,
        'cmap': 'viridis',
        'vmin': 0.0,
        'vmax': 200.0,
        'mask_var': 'elev',
        'mask_threshold': -1.1869,
    },
]

SATELLITE_FRAME_DIR = FRAME_OUTPUT_DIR / 'satellite_vs_model_chla'
SATELLITE_CHLA_VMIN = 0.0
SATELLITE_CHLA_VMAX = 30.0
SATELLITE_CHLA_CMAP = 'viridis'
MAP_EXTENT_LONLAT = (4.27, 7.6, 52.47, 53.6)
USE_BASEMAP = False
BASEMAP_SOURCE = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}'
BASEMAP_ZOOM = 8
SURFACE_ALPHA = 0.72

RUN_DEFAULT_WORKFLOW = False
RUN_SPINUP_COMPARISON = False


In [ ]:
# -----------------------------------------------------------------------------
# HELPER FUNCTIONS
# -----------------------------------------------------------------------------


def _apply_theme() -> None:
    sns.set_theme(style='ticks')
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'font.size': 11,
        'axes.titlesize': 13,
        'axes.labelsize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.edgecolor': 'black',
        'axes.linewidth': 1.0,
        'figure.facecolor': 'white',
        'savefig.facecolor': 'white',
    })


def _year_pattern(year: int) -> str:
    return NC_PATTERN.format(year=year)


def _find_time_dim(da: xr.DataArray) -> str:
    for dim in da.dims:
        if 'time' in dim.lower():
            return dim
    raise ValueError(f'No time dimension found in {da.dims}')


def _find_dataset_time_coord(ds: xr.Dataset) -> str:
    for name in list(ds.coords) + list(ds.dims):
        if 'time' in str(name).lower():
            return str(name)
    raise ValueError('No time coordinate found in dataset.')


def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for dim in da.dims:
        if dim == time_dim:
            continue
        if any(token in dim.lower() for token in candidates):
            return dim
    return None


def _find_xy_dims(da: xr.DataArray, excluded_dims: tuple[str, ...]) -> tuple[str, str]:
    remain = [dim for dim in da.dims if dim not in excluded_dims]
    if len(remain) != 2:
        raise ValueError(f'Expected 2 horizontal dims, got {remain} from {da.dims}')
    return remain[0], remain[1]


def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da


def _drop_duplicate_dataset_time(ds: xr.Dataset) -> xr.Dataset:
    time_name = _find_dataset_time_coord(ds)
    time_values = pd.DatetimeIndex(pd.to_datetime(ds[time_name].values))
    if time_values.has_duplicates:
        ds = ds.isel({time_name: ~time_values.duplicated()})
    return ds


def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables or name in ds.coords:
            return name
    return None


def _to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)


def _safe_fill_coords(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=float)
    arr[~np.isfinite(arr)] = np.nan
    df = pd.DataFrame(arr)
    return df.ffill(axis=1).bfill(axis=1).ffill(axis=0).bfill(axis=0).to_numpy(dtype=float)


def _format_timestamp_for_filename(value) -> str:
    return pd.Timestamp(value).strftime('%Y-%m-%dT%H-%M-%S')


def _subdomain_label(subdomain: dict | None) -> str:
    if subdomain is None:
        return 'full domain'
    return (
        f"{subdomain.get('name', 'subdomain')} "
        f"(y={subdomain['y_slice'][0]}:{subdomain['y_slice'][1]}, "
        f"x={subdomain['x_slice'][0]}:{subdomain['x_slice'][1]})"
    )


def _validate_subdomain(da: xr.DataArray, subdomain: dict, excluded_dims: tuple[str, ...]) -> tuple[str, str]:
    y_dim, x_dim = _find_xy_dims(da, excluded_dims)
    y0, y1 = subdomain['y_slice']
    x0, x1 = subdomain['x_slice']
    if not (0 <= y0 < y1 <= da.sizes[y_dim]):
        raise IndexError(f"y_slice={subdomain['y_slice']} out of bounds for {y_dim} size {da.sizes[y_dim]}")
    if not (0 <= x0 < x1 <= da.sizes[x_dim]):
        raise IndexError(f"x_slice={subdomain['x_slice']} out of bounds for {x_dim} size {da.sizes[x_dim]}")
    return y_dim, x_dim


def _subset_subdomain(da: xr.DataArray, subdomain: dict | None, excluded_dims: tuple[str, ...]) -> tuple[xr.DataArray, tuple[str, str]]:
    y_dim, x_dim = _find_xy_dims(da, excluded_dims)
    if subdomain is None:
        return da, (y_dim, x_dim)
    y_dim, x_dim = _validate_subdomain(da, subdomain, excluded_dims)
    y0, y1 = subdomain['y_slice']
    x0, x1 = subdomain['x_slice']
    return da.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)}), (y_dim, x_dim)


def _resolve_layer(da: xr.DataArray, layer_index: int | None, time_dim: str) -> tuple[xr.DataArray, str | None]:
    z_dim = _find_vertical_dim(da, time_dim)
    if z_dim is None:
        if layer_index is not None:
            return da, None
        return da, None
    if layer_index is None:
        raise ValueError(f'layer_index must be provided for variable with vertical dim {z_dim}.')
    if layer_index >= da.sizes[z_dim] or layer_index < -da.sizes[z_dim]:
        raise IndexError(f'layer_index={layer_index} out of bounds for {z_dim} size {da.sizes[z_dim]}')
    return da.isel({z_dim: layer_index}), z_dim


def _build_horizontal_coords(
    ds: xr.Dataset,
    y_dim: str,
    x_dim: str,
    *,
    fill_invalid: bool = False,
) -> tuple[np.ndarray, np.ndarray, bool]:
    lon_name = _pick_coord_name(ds, ('lonc', 'lon', 'longitude'))
    lat_name = _pick_coord_name(ds, ('latc', 'lat', 'latitude'))
    if lon_name is not None and lat_name is not None:
        lon_da = ds[lon_name]
        lat_da = ds[lat_name]
        if y_dim in lon_da.dims and x_dim in lon_da.dims and y_dim in lat_da.dims and x_dim in lat_da.dims:
            x_plot = _to_float(lon_da.transpose(y_dim, x_dim).values)
            y_plot = _to_float(lat_da.transpose(y_dim, x_dim).values)
            if fill_invalid:
                x_plot = _safe_fill_coords(x_plot)
                y_plot = _safe_fill_coords(y_plot)
                return x_plot, y_plot, True
            if np.isfinite(x_plot).all() and np.isfinite(y_plot).all():
                return x_plot, y_plot, True
    ny = ds.sizes[y_dim]
    nx = ds.sizes[x_dim]
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))
    return x_plot, y_plot, False


def _maybe_smooth(
    series: xr.DataArray,
    time_dim: str,
    *,
    use_daily_mean: bool = False,
    rolling_window: int | None = None,
) -> xr.DataArray:
    out = series
    if use_daily_mean:
        out = out.resample({time_dim: '1D'}).mean(skipna=True)
    if rolling_window is not None:
        if rolling_window < 1:
            raise ValueError('rolling_window must be >= 1 or None')
        out = out.rolling({time_dim: rolling_window}, center=True).mean()
    return out


def _domain_mean_series(
    da: xr.DataArray,
    *,
    layer_index: int | None,
    subdomain: dict | None = None,
    use_daily_mean: bool = False,
    rolling_window: int | None = None,
    mask_negative: bool = False,
) -> tuple[xr.DataArray, str]:
    da = da.squeeze(drop=True)
    time_dim = _find_time_dim(da)
    da, z_dim = _resolve_layer(da, layer_index, time_dim)
    excluded = (time_dim,) if z_dim is None else (time_dim,)
    da, _ = _subset_subdomain(da, subdomain, excluded)
    da = _drop_duplicate_time(da, time_dim)
    spatial_dims = [dim for dim in da.dims if dim != time_dim]
    if not spatial_dims:
        raise ValueError('No spatial dimensions left for averaging.')
    if mask_negative:
        da = da.where(da >= 0)
    series = da.mean(dim=spatial_dims, skipna=True)
    series = _maybe_smooth(
        series,
        time_dim,
        use_daily_mean=use_daily_mean,
        rolling_window=rolling_window,
    )
    return series, time_dim


def _lonlat_to_webmercator(lon: np.ndarray, lat: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    radius = 6378137.0
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)
    lat = np.clip(lat, -85.05112878, 85.05112878)
    x = radius * np.deg2rad(lon)
    y = radius * np.log(np.tan(np.pi / 4.0 + np.deg2rad(lat) / 2.0))
    return x, y


def _nearest_time_indices(reference_times: np.ndarray, query_times: np.ndarray) -> np.ndarray:
    right = np.searchsorted(reference_times, query_times, side='left')
    left = np.clip(right - 1, 0, len(reference_times) - 1)
    right = np.clip(right, 0, len(reference_times) - 1)
    left_diff = np.abs(query_times - reference_times[left]).astype(np.int64)
    right_diff = np.abs(reference_times[right] - query_times).astype(np.int64)
    use_right = right_diff < left_diff
    return np.where(use_right, right, left)


def open_spinup_dataset(spinup_name: str = DEFAULT_SPINUP, year: int = YEAR) -> xr.Dataset:
    data_dir = BASE_OUTPUT_DIR / spinup_name
    if not data_dir.exists():
        raise FileNotFoundError(f'Spinup directory does not exist: {data_dir}')
    files = sorted(data_dir.glob(_year_pattern(year)))
    if not files:
        raise FileNotFoundError(f'No files found for {spinup_name}: {_year_pattern(year)}')
    ds = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )
    return _drop_duplicate_dataset_time(ds)


def load_measurements(csv_path: Path = MEASUREMENT_FILE) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV file not found: {csv_path}')
    df = pd.read_csv(csv_path, na_values=['NA', ''], parse_dates=['timestamp'])
    if 'timestamp' not in df.columns:
        raise KeyError("Column 'timestamp' not found in observations CSV.")
    return df.sort_values('timestamp').reset_index(drop=True)


def load_primary_production_observations(
    csv_path: Path = PRIMARY_PRODUCTION_FILE,
    *,
    year: int = YEAR,
) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV file not found: {csv_path}')
    df = pd.read_csv(csv_path, na_values=['NA', ''])
    if 'Date' not in df.columns or 'Daily_PP' not in df.columns:
        raise KeyError("CSV must contain columns 'Date' and 'Daily_PP'.")
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    out = df[['Date', 'Daily_PP']].dropna(subset=['Date', 'Daily_PP']).copy()
    out = out[out['Date'].dt.year == year].sort_values('Date').reset_index(drop=True)
    if out.empty:
        raise ValueError(f'No Daily_PP observations found for {year}.')
    return out


In [ ]:
# -----------------------------------------------------------------------------
# ANALYSIS AND PLOTTING
# -----------------------------------------------------------------------------


def plot_domain_mean_trends(
    ds: xr.Dataset,
    variables: list[str],
    *,
    layer_index: int = MODEL_SURFACE_LAYER_INDEX,
    use_daily_mean: bool = False,
) -> None:
    _apply_theme()
    nvars = len(variables)
    ncols = 4
    nrows = int(np.ceil(nvars / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 2.8 * nrows), sharex=True)
    axes = np.atleast_1d(axes).ravel()

    missing_vars: list[str] = []
    failed_vars: list[tuple[str, str]] = []

    for i, var_name in enumerate(variables):
        ax = axes[i]
        try:
            if var_name not in ds.variables:
                missing_vars.append(var_name)
                ax.text(0.5, 0.5, f'{var_name}\nnot found', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(var_name)
                ax.set_axis_off()
                continue

            series, time_dim = _domain_mean_series(
                ds[var_name],
                layer_index=layer_index,
                subdomain=None,
                use_daily_mean=use_daily_mean,
                mask_negative=var_name in NONNEGATIVE_VARS,
            )
            ax.plot(series[time_dim].values, series.values, lw=1.6)
            units = ds[var_name].attrs.get('units', '')
            ax.set_title(var_name)
            ax.set_ylabel(f'{var_name} [{units}]' if units else var_name, fontsize=8)
            ax.grid(True, alpha=0.3)
        except Exception as exc:
            failed_vars.append((var_name, str(exc)))
            ax.text(0.5, 0.5, f'{var_name}\nfailed', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(var_name)
            ax.set_axis_off()

    for j in range(nvars, len(axes)):
        axes[j].set_visible(False)
    for ax in axes[:nvars]:
        ax.tick_params(axis='x', labelrotation=35)

    fig.suptitle(f'{YEAR} domain-mean trends | {DEFAULT_SPINUP}', y=1.02)
    plt.tight_layout()
    plt.show()

    if missing_vars:
        print('Variables not found in dataset:')
        print(missing_vars)
    if failed_vars:
        print('Variables that failed during processing:')
        for name, err in failed_vars:
            print(f'- {name}: {err}')


def export_map_frames(
    ds: xr.Dataset,
    *,
    plotvar: str,
    output_dir: Path,
    layer_index: int | None,
    time_step: int,
    cmap: str,
    vmin: float,
    vmax: float,
    mask_var: str | None = None,
    mask_threshold: float | None = None,
) -> None:
    _apply_theme()
    if plotvar not in ds.variables:
        raise KeyError(f"Variable '{plotvar}' not found. Available: {sorted(ds.data_vars)}")
    if vmin is None or vmax is None:
        raise ValueError('vmin and vmax must both be provided for consistent frame export.')

    da = ds[plotvar].squeeze(drop=True)
    time_dim = _find_time_dim(da)
    field, z_dim = _resolve_layer(da, layer_index, time_dim)
    field = _drop_duplicate_time(field, time_dim)
    y_dim, x_dim = _find_xy_dims(field, (time_dim,))
    x_plot, y_plot, use_geo = _build_horizontal_coords(ds, y_dim, x_dim)
    layer_text = '' if z_dim is None else f' | layer {layer_index}'

    if mask_var is not None:
        if mask_var not in ds.variables:
            raise KeyError(f"Mask variable '{mask_var}' not found.")
        mask_da = ds[mask_var].squeeze(drop=True)
        mask_time_dim = _find_time_dim(mask_da)
        mask_da = _drop_duplicate_time(mask_da, mask_time_dim)
        field, mask_da = xr.align(field, mask_da, join='inner')
    else:
        mask_da = None

    frame_indices = np.arange(0, field.sizes[time_dim], time_step, dtype=int)
    if frame_indices.size == 0:
        raise ValueError('No frames selected. Check time_step and time length.')

    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'Exporting {frame_indices.size} frames to: {output_dir}')

    for frame_number, frame_idx in enumerate(frame_indices):
        frame2d = _to_float(field.isel({time_dim: int(frame_idx)}).transpose(y_dim, x_dim).values)
        if mask_da is not None:
            wet_mask = _to_float(mask_da.isel({time_dim: int(frame_idx)}).transpose(y_dim, x_dim).values) > mask_threshold
            frame2d[~wet_mask] = np.nan

        timestamp = pd.Timestamp(field[time_dim].values[int(frame_idx)])
        frame_name = f"frame_{frame_number:04d}_{_format_timestamp_for_filename(timestamp)}.png"
        frame_path = output_dir / frame_name

        fig, ax = plt.subplots(figsize=(8, 6))
        mesh = ax.pcolormesh(
            x_plot,
            y_plot,
            np.ma.masked_invalid(frame2d),
            shading='auto',
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )
        cbar = fig.colorbar(mesh, ax=ax, fraction=0.035, pad=0.03)
        units = da.attrs.get('units', '')
        cbar.set_label(f'{plotvar} [{units}]' if units else plotvar)
        ax.set_title(f'{plotvar}{layer_text} | {timestamp}')
        ax.set_xlabel('Longitude' if use_geo else x_dim)
        ax.set_ylabel('Latitude' if use_geo else y_dim)
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        fig.savefig(frame_path, dpi=200, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved {frame_indices.size} frames: {output_dir}')


def plot_subdomain_location(
    ds: xr.Dataset,
    subdomain: dict = SUBDOMAIN,
    *,
    bathy_var: str = 'bathymetry',
) -> None:
    _apply_theme()
    if bathy_var not in ds.variables:
        candidates = ('bathymetry', 'depth', 'h', 'H', 'bathy', 'bat', 'topo', 'd', 'water_depth')
        bathy_var = next((name for name in candidates if name in ds.variables), None)
        if bathy_var is None:
            raise KeyError('Bathymetry variable not found in dataset.')

    bathy = ds[bathy_var].squeeze(drop=True)
    if len(bathy.dims) == 3:
        time_dim = _find_time_dim(bathy)
        bathy = bathy.isel({time_dim: 0})
    if len(bathy.dims) != 2:
        raise ValueError(f'Expected 2D bathymetry, got dims {bathy.dims}')

    y_dim, x_dim = bathy.dims
    _validate_subdomain(bathy, subdomain, ())
    x_plot, y_plot, use_geo = _build_horizontal_coords(ds, y_dim, x_dim)
    bathy2d = _to_float(bathy.transpose(y_dim, x_dim).values)
    y0, y1 = subdomain['y_slice']
    x0, x1 = subdomain['x_slice']

    fig, ax = plt.subplots(figsize=(8.6, 6.8))
    mesh = ax.pcolormesh(x_plot, y_plot, np.ma.masked_invalid(bathy2d), shading='auto', cmap='cividis')
    cbar = fig.colorbar(mesh, ax=ax, fraction=0.04, pad=0.03)
    units = bathy.attrs.get('units', '')
    cbar.set_label(f'{bathy_var} [{units}]' if units else bathy_var)

    if use_geo:
        x_poly = [x_plot[y0, x0], x_plot[y0, x1 - 1], x_plot[y1 - 1, x1 - 1], x_plot[y1 - 1, x0], x_plot[y0, x0]]
        y_poly = [y_plot[y0, x0], y_plot[y0, x1 - 1], y_plot[y1 - 1, x1 - 1], y_plot[y1 - 1, x0], y_plot[y0, x0]]
    else:
        x_poly = [x0, x1, x1, x0, x0]
        y_poly = [y0, y0, y1, y1, y0]
    ax.plot(x_poly, y_poly, color='red', lw=2.2, label=subdomain.get('name', 'Selected subdomain'))

    ax.set_title('Subdomain location on bathymetry map')
    ax.set_xlabel('Longitude' if use_geo else x_dim)
    ax.set_ylabel('Latitude' if use_geo else y_dim)
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()


def plot_subdomain_dynamics(
    ds: xr.Dataset,
    *,
    chla_var: str = 'Chla',
    elev_var: str = 'elev',
    subdomain: dict = SUBDOMAIN,
    layer_indices: dict[str, int] = CHLA_DIAGNOSTIC_LAYERS,
    use_daily_mean: bool = False,
    rolling_window: int | None = None,
) -> None:
    _apply_theme()
    if chla_var not in ds.variables or elev_var not in ds.variables:
        raise KeyError('Required variables not found in dataset.')

    chla = ds[chla_var].squeeze(drop=True)
    chla_time_dim = _find_time_dim(chla)
    chla_z_dim = _find_vertical_dim(chla, chla_time_dim)
    if chla_z_dim is None:
        raise ValueError(f'No vertical dimension found for {chla_var}.')
    chla, (y_dim, x_dim) = _subset_subdomain(chla, subdomain, (chla_time_dim, chla_z_dim))

    layer_series = {}
    for label, layer_idx in layer_indices.items():
        if layer_idx >= chla.sizes[chla_z_dim] or layer_idx < -chla.sizes[chla_z_dim]:
            raise IndexError(f'Layer index {layer_idx} out of bounds for {chla_z_dim}')
        series = chla.isel({chla_z_dim: layer_idx}).mean(dim=(y_dim, x_dim), skipna=True)
        series = _drop_duplicate_time(series, chla_time_dim)
        series = _maybe_smooth(series, chla_time_dim, use_daily_mean=use_daily_mean, rolling_window=rolling_window)
        layer_series[label] = series.where(series >= 0)

    elev = ds[elev_var].squeeze(drop=True)
    elev_time_dim = _find_time_dim(elev)
    elev, (elev_y_dim, elev_x_dim) = _subset_subdomain(elev, subdomain, (elev_time_dim,))
    elev_series = elev.mean(dim=(elev_y_dim, elev_x_dim), skipna=True)
    elev_series = _drop_duplicate_time(elev_series, elev_time_dim)
    elev_series = _maybe_smooth(elev_series, elev_time_dim, use_daily_mean=use_daily_mean, rolling_window=rolling_window)

    fig, ax1 = plt.subplots(figsize=(12, 4.8))
    ax2 = ax1.twinx()
    styles = {'center': '-', 'upper': '--', 'lower': ':'}
    for label, series in layer_series.items():
        layer_idx = layer_indices[label]
        ax1.plot(
            series[chla_time_dim].values,
            series.values,
            lw=2.0 if label == 'center' else 1.4,
            ls=styles.get(label, '-'),
            label=f'{chla_var} {label} layer ({layer_idx})',
        )
    ax2.plot(elev_series[elev_time_dim].values, elev_series.values, color='black', lw=1.3, alpha=0.65, label=elev_var)

    chla_units = chla.attrs.get('units', '')
    elev_units = elev.attrs.get('units', '')
    ax1.set_ylabel(f'{chla_var} [{chla_units}]' if chla_units else chla_var)
    ax2.set_ylabel(f'{elev_var} [{elev_units}]' if elev_units else elev_var)
    ax1.set_xlabel('Time')
    ax1.set_title(f'Subdomain dynamics | {_subdomain_label(subdomain)}')
    ax1.grid(True, alpha=0.25)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', ncol=2, fontsize=9)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def plot_model_vs_observations(
    ds: xr.Dataset,
    measurement_df: pd.DataFrame,
    *,
    var_map: dict[str, str] = OBS_VAR_MAP,
    subdomain: dict = SUBDOMAIN,
    layer_index: int = MODEL_SURFACE_LAYER_INDEX,
    year: int = YEAR,
    use_daily_mean: bool = False,
    rolling_window: int | None = None,
) -> None:
    _apply_theme()
    obs_df = measurement_df.dropna(subset=['timestamp']).copy()
    obs_df = obs_df[obs_df['timestamp'].dt.year == year].sort_values('timestamp')
    if obs_df.empty:
        raise ValueError(f'No observations found for {year}.')

    n_panels = len(var_map)
    ncols = 2
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows), sharex=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, (model_var, obs_col) in zip(axes, var_map.items()):
        if obs_col not in obs_df.columns:
            ax.text(0.5, 0.5, f"Obs column '{obs_col}' not found", ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue
        if model_var not in ds.variables:
            ax.text(0.5, 0.5, f"Model variable '{model_var}' not found", ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue

        model_series, time_dim = _domain_mean_series(
            ds[model_var],
            layer_index=layer_index,
            subdomain=subdomain,
            use_daily_mean=use_daily_mean,
            rolling_window=rolling_window,
            mask_negative=model_var in NONNEGATIVE_VARS,
        )
        model_df = pd.DataFrame({
            'timestamp': pd.to_datetime(model_series[time_dim].values),
            'value': model_series.values,
        })
        model_df = model_df[model_df['timestamp'].dt.year == year]
        obs_var = obs_df[['timestamp', obs_col]].dropna()

        ax.plot(model_df['timestamp'], model_df['value'], lw=2.0, color='tab:blue', label=f'Model {model_var}')
        ax.scatter(obs_var['timestamp'], obs_var[obs_col], s=18, color='tab:orange', alpha=0.85, label=f'Obs {obs_col}', zorder=3)
        units = ds[model_var].attrs.get('units', '')
        ax.set_ylabel(f'{model_var} / {obs_col} [{units}]' if units else f'{model_var} / {obs_col}')
        ax.set_title(f'{model_var} (model) vs {obs_col} (obs)')
        ax.grid(True, alpha=0.25)
        ax.legend(loc='upper left', fontsize=8)

    for ax in axes[n_panels:]:
        ax.set_visible(False)
    for ax in axes[:n_panels]:
        ax.set_xlabel('Time')

    fig.suptitle(f'Model vs observed surface concentrations in {year} | {_subdomain_label(subdomain)}', y=1.02)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def plot_primary_production_vs_obs(
    ds: xr.Dataset,
    pp_obs: pd.DataFrame,
    *,
    model_var: str = 'netPPm2',
    subdomain: dict = SUBDOMAIN,
    year: int = YEAR,
) -> None:
    _apply_theme()
    if model_var not in ds.variables:
        raise KeyError(f"Variable '{model_var}' not found. Available: {sorted(ds.data_vars)}")

    model_series, time_dim = _domain_mean_series(
        ds[model_var],
        layer_index=None,
        subdomain=subdomain,
        use_daily_mean=True,
        mask_negative=True,
    )
    model_df = pd.DataFrame({
        'Date': pd.to_datetime(model_series[time_dim].values),
        'Model_PP': model_series.values,
    })
    model_df = model_df.dropna(subset=['Date', 'Model_PP'])
    model_df = model_df[model_df['Date'].dt.year == year]
    if model_df.empty:
        raise ValueError(f'No model {model_var} values found for {year}.')

    fig, ax = plt.subplots(figsize=(12, 4.8))
    ax.plot(model_df['Date'], model_df['Model_PP'], lw=2.0, color='tab:blue', label=f'Model {model_var} (daily mean)')
    ax.scatter(pp_obs['Date'], pp_obs['Daily_PP'], s=24, color='tab:orange', alpha=0.9, label='Observed Daily_PP', zorder=3)
    units = ds[model_var].attrs.get('units', '')
    ax.set_ylabel(f'PP [{units}]' if units else 'PP')
    ax.set_xlabel('Time')
    ax.set_title(f'Model vs observed primary production in {year} | {_subdomain_label(subdomain)}')
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper left')
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def export_satellite_vs_model_frames(
    model_ds: xr.Dataset,
    *,
    satellite_path: Path = SATELLITE_FILE,
    output_dir: Path = SATELLITE_FRAME_DIR,
    model_var: str = 'Chla',
    satellite_var: str = 'CHL',
    surface_layer_index: int = MODEL_SURFACE_LAYER_INDEX,
    time_step: int = 1,
    cmap: str = SATELLITE_CHLA_CMAP,
    vmin: float = SATELLITE_CHLA_VMIN,
    vmax: float = SATELLITE_CHLA_VMAX,
    map_extent_lonlat: tuple[float, float, float, float] | None = MAP_EXTENT_LONLAT,
    use_basemap: bool = USE_BASEMAP,
) -> None:
    _apply_theme()
    if model_var not in model_ds.variables:
        raise KeyError(f"Model variable '{model_var}' not found.")
    if not satellite_path.exists():
        raise FileNotFoundError(f'Satellite file not found: {satellite_path}')

    if use_basemap:
        try:
            import contextily as ctx
        except ImportError as exc:
            raise ImportError('Basemap requested but contextily is not installed.') from exc
    else:
        ctx = None

    with xr.open_dataset(satellite_path, decode_times=True) as sat_ds:
        sat_da = sat_ds[satellite_var].squeeze(drop=True)
        sat_time_dim = _find_time_dim(sat_da)
        sat_da = _drop_duplicate_time(sat_da, sat_time_dim)
        sat_times = pd.to_datetime(sat_da[sat_time_dim].values)

        sat_lon_name = _pick_coord_name(sat_ds, ('longitude', 'lon'))
        sat_lat_name = _pick_coord_name(sat_ds, ('latitude', 'lat'))
        if sat_lon_name is None or sat_lat_name is None:
            raise KeyError('Cannot find satellite lon/lat coordinates.')
        sat_lon1d = np.asarray(sat_ds[sat_lon_name].values, dtype=float)
        sat_lat1d = np.asarray(sat_ds[sat_lat_name].values, dtype=float)
        sat_lon2d, sat_lat2d = np.meshgrid(sat_lon1d, sat_lat1d)
        sat_x_map, sat_y_map = _lonlat_to_webmercator(sat_lon2d, sat_lat2d)

        model_da = model_ds[model_var].squeeze(drop=True)
        model_time_dim = _find_time_dim(model_da)
        model_surface, model_z_dim = _resolve_layer(model_da, surface_layer_index, model_time_dim)
        if model_z_dim is None:
            raise ValueError(f'Model variable {model_var} must have a vertical dimension for surface comparison.')
        model_surface = _drop_duplicate_time(model_surface, model_time_dim)
        model_y_dim, model_x_dim = _find_xy_dims(model_surface, (model_time_dim,))

        lon_plot, lat_plot, _ = _build_horizontal_coords(model_ds, model_y_dim, model_x_dim, fill_invalid=True)
        model_x_map, model_y_map = _lonlat_to_webmercator(lon_plot, lat_plot)
        model_times = pd.to_datetime(model_surface[model_time_dim].values)

        sat_times_np = sat_times.to_numpy(dtype='datetime64[ns]')
        model_times_np = model_times.to_numpy(dtype='datetime64[ns]')
        sat_frame_idx = np.arange(0, len(sat_times_np), time_step, dtype=int)
        model_frame_idx = _nearest_time_indices(model_times_np, sat_times_np[sat_frame_idx])

        if map_extent_lonlat is not None:
            x_extent, y_extent = _lonlat_to_webmercator(
                np.array([map_extent_lonlat[0], map_extent_lonlat[1]], dtype=float),
                np.array([map_extent_lonlat[2], map_extent_lonlat[3]], dtype=float),
            )
            x_limits = (float(x_extent[0]), float(x_extent[1]))
            y_limits = (float(y_extent[0]), float(y_extent[1]))
        else:
            x_limits = (float(np.nanmin(model_x_map)), float(np.nanmax(model_x_map)))
            y_limits = (float(np.nanmin(model_y_map)), float(np.nanmax(model_y_map)))

        output_dir.mkdir(parents=True, exist_ok=True)
        print(f'Exporting {len(sat_frame_idx)} satellite/model comparison frames to: {output_dir}')

        for frame_number, (sat_idx, mod_idx) in enumerate(zip(sat_frame_idx, model_frame_idx)):
            sat_frame = np.asarray(sat_da.isel({sat_time_dim: int(sat_idx)}).values, dtype=float)
            if sat_frame.ndim > 2:
                sat_frame = sat_frame.squeeze()
            mod_frame = _to_float(
                model_surface.isel({model_time_dim: int(mod_idx)}).transpose(model_y_dim, model_x_dim).values
            )
            mod_frame = np.where(mod_frame < 0, np.nan, mod_frame)

            sat_stamp = pd.Timestamp(sat_times[sat_idx])
            mod_stamp = pd.Timestamp(model_times[mod_idx])
            frame_name = (
                f'frame_{frame_number:04d}_sat_{_format_timestamp_for_filename(sat_stamp)}'
                f'_model_{_format_timestamp_for_filename(mod_stamp)}.png'
            )
            frame_path = output_dir / frame_name

            fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
            plt.subplots_adjust(wspace=0.04, right=0.88)
            cbar_ax = fig.add_axes([0.90, 0.12, 0.018, 0.74])
            sm = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
            sm.set_array([])
            fig.colorbar(sm, cax=cbar_ax, label='Chla / CHL (mg m⁻³)')

            for ax in axes:
                ax.set_xlim(*x_limits)
                ax.set_ylim(*y_limits)
                ax.tick_params(labelbottom=False, labelleft=False)
                if ctx is not None:
                    ctx.add_basemap(
                        ax,
                        source=BASEMAP_SOURCE,
                        crs='EPSG:3857',
                        zoom=BASEMAP_ZOOM,
                        attribution='',
                        zorder=1,
                        reset_extent=False,
                    )

            axes[0].pcolormesh(
                sat_x_map,
                sat_y_map,
                np.ma.masked_invalid(sat_frame),
                shading='auto',
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                alpha=SURFACE_ALPHA,
                zorder=3,
            )
            axes[0].set_title(f'Satellite CHL | {sat_stamp.date()}', fontsize=11)

            axes[1].pcolormesh(
                model_x_map,
                model_y_map,
                np.ma.masked_invalid(mod_frame),
                shading='auto',
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                alpha=SURFACE_ALPHA,
                zorder=3,
            )
            axes[1].set_title(f'Model Chla (surface) | {mod_stamp.date()}', fontsize=11)

            fig.savefig(frame_path, dpi=200, bbox_inches='tight')
            plt.close(fig)

        print(f'Saved {len(sat_frame_idx)} comparison frames: {output_dir}')


def plot_spinup_comparison(
    spinup_names: list[str] = SPINUP_NAMES,
    *,
    variables: list[str] = TREND_VARIABLES,
    year: int = YEAR,
    layer_index: int = MODEL_SURFACE_LAYER_INDEX,
    use_daily_mean: bool = False,
) -> None:
    _apply_theme()
    colors = sns.color_palette('husl', len(spinup_names)).as_hex()
    datasets: dict[str, xr.Dataset] = {}

    try:
        for spinup in spinup_names:
            try:
                datasets[spinup] = open_spinup_dataset(spinup, year)
                print(f'Loaded {spinup}')
            except FileNotFoundError as exc:
                warnings.warn(str(exc))

        if not datasets:
            raise FileNotFoundError('No spinup datasets could be loaded.')

        nvars = len(variables)
        ncols = 4
        nrows = int(np.ceil(nvars / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 2.8 * nrows), sharex=True)
        axes = np.atleast_1d(axes).ravel()

        missing_vars: list[str] = []
        failed_vars: list[tuple[str, str, str]] = []

        for i, var_name in enumerate(variables):
            ax = axes[i]
            plotted = False
            for (spinup, ds), color in zip(datasets.items(), colors):
                try:
                    if var_name not in ds.variables:
                        continue
                    series, time_dim = _domain_mean_series(
                        ds[var_name],
                        layer_index=layer_index,
                        subdomain=None,
                        use_daily_mean=use_daily_mean,
                        mask_negative=var_name in NONNEGATIVE_VARS,
                    )
                    ax.plot(series[time_dim].values, series.values, lw=1.6, color=color, label=spinup)
                    plotted = True
                except Exception as exc:
                    failed_vars.append((var_name, spinup, str(exc)))

            if not plotted:
                missing_vars.append(var_name)
                ax.text(0.5, 0.5, f'{var_name}\nnot found', ha='center', va='center', transform=ax.transAxes)
                ax.set_axis_off()
                continue

            units = ''
            for ds in datasets.values():
                if var_name in ds.variables:
                    units = ds[var_name].attrs.get('units', '')
                    break
            ax.set_title(var_name)
            ax.set_ylabel(f'{var_name} [{units}]' if units else var_name, fontsize=8)
            ax.grid(True, alpha=0.3)

        for j in range(nvars, len(axes)):
            axes[j].set_visible(False)
        for ax in axes[:nvars]:
            ax.tick_params(axis='x', labelrotation=35)

        handles = [plt.Line2D([0], [0], color=color, lw=1.6, label=spinup) for spinup, color in zip(datasets.keys(), colors)]
        fig.legend(handles=handles, loc='lower right', fontsize=10, framealpha=0.9)
        fig.suptitle(f'{year} domain-mean trends — spinup comparison', y=1.02)
        plt.tight_layout()
        plt.show()

        if missing_vars:
            print('Variables not found in any dataset:', missing_vars)
        if failed_vars:
            print('Variables that failed during processing:')
            for name, spinup, err in failed_vars:
                print(f'  - {name} ({spinup}): {err}')
    finally:
        for ds in datasets.values():
            ds.close()


In [ ]:
# -----------------------------------------------------------------------------
# ENTRY POINT
# -----------------------------------------------------------------------------


def main() -> None:
    print(f'Opening {DEFAULT_SPINUP} for {YEAR} …')
    ds = open_spinup_dataset(DEFAULT_SPINUP, YEAR)
    try:
        print('Plotting domain-mean trends …')
        plot_domain_mean_trends(ds, TREND_VARIABLES)

        print('Exporting map frames …')
        for frame_config in FRAME_EXPORTS:
            export_map_frames(ds, **frame_config)

        print('Plotting subdomain location …')
        plot_subdomain_location(ds, SUBDOMAIN)

        print('Plotting subdomain dynamics …')
        plot_subdomain_dynamics(ds, subdomain=SUBDOMAIN)

        print('Loading observations …')
        measurement_df = load_measurements()
        display(measurement_df.head())

        print('Plotting model vs observations …')
        plot_model_vs_observations(ds, measurement_df, subdomain=SUBDOMAIN)

        print('Loading primary production observations …')
        pp_obs = load_primary_production_observations()

        print('Plotting primary production comparison …')
        plot_primary_production_vs_obs(ds, pp_obs, subdomain=SUBDOMAIN)

        if SATELLITE_FILE.exists():
            print('Exporting satellite vs model frames …')
            export_satellite_vs_model_frames(model_ds=ds)
        else:
            print(f'Skipping satellite comparison because the file was not found: {SATELLITE_FILE}')

        print(f'All outputs written under: {FRAME_OUTPUT_DIR.parent}')
    finally:
        ds.close()


if RUN_DEFAULT_WORKFLOW:
    main()

if RUN_SPINUP_COMPARISON:
    plot_spinup_comparison()


In [ ]:
plot_spinup_comparison()